# Near & Far Transfer

**Track:** Learning
**Construct:** Structural transfer and abstraction

Tests whether models can genuinely transfer learned structure to novel contexts, requiring actual abstraction rather than instruction-following.

## Cognitive Science Background

When a student learns to solve quadratic equations in math class, can they apply that same reasoning to physics problems? This question — whether knowledge acquired in one context transfers to another — has been central to learning science since Thorndike & Woodworth (1901) first showed that training in one skill rarely transfers automatically to related skills.

**Transfer** exists on a spectrum. **Near transfer** occurs between similar contexts — like applying addition skills to slightly different number formats. **Far transfer** requires abstracting the underlying structure and mapping it onto a completely different domain — like recognizing that supply-and-demand curves follow the same equilibrium logic as chemical reaction rates. Barnett & Ceci (2002) formalized this distinction, showing that far transfer is rare in humans and requires deep structural understanding rather than surface pattern matching.

Anderson's ACT* framework (1987) explains why: **procedural transfer** (applying the same steps) is relatively easy, but **declarative transfer** (abstracting the principle and re-deriving steps for a new domain) requires genuine comprehension. Most learning systems — human and artificial — default to procedural transfer and fail when the surface features change.

This benchmark tests whether LLMs can perform genuine structural transfer, or whether they merely follow explicit instructions. If all rules are given, any capable model scores perfectly — that's instruction-following, not transfer. Real transfer means inducing the underlying structure from examples and applying it where rules are withheld.

## Methodology

Four conditions test progressively deeper transfer, with rules deliberately omitted to force genuine abstraction:


| Condition | Weight | Description |
|------|--------|-------------|
| Identical | 0.15 | Same system, all rules, held-out items — baseline |
| Near transfer | 0.25 | Same domain, 1 rule omitted — must infer from context |
| Far transfer | 0.30 | Different domain, no rules — only 2 worked examples |
| Zero-shot structural | 0.30 | Stateful system, no rules — only 1 worked example |


## Scoring

$$\text{Score} = 0.15 \times \text{identical} + 0.25 \times \text{near} + 0.30 \times \text{far} + 0.30 \times \text{zero-shot}$$

| Score | Interpretation |
|:---:|---|
| 0.8–1.0 | Deep structural transfer — abstracts rules from minimal examples across domains |
| 0.5–0.8 | Good — near transfer works but far transfer is inconsistent |
| 0.2–0.5 | Surface-level — copies patterns but doesn't abstract structure |
| 0.0–0.2 | No transfer — performance collapses without explicit rules |

### References

Thorndike & Woodworth (1901), Barnett & Ceci (2002), Anderson (1987)

In [ ]:
!pip install -q protobuf==5.29.6 kaggle-benchmarks numpy 2>/dev/null


In [ ]:
"""
Novel Rule System Generator for Learning Benchmarks.

Generates procedural rule systems that cannot be in training data.
Each system defines a mapping from inputs to outputs via a chain
of deterministic rules. Difficulty is controlled by:
- Number of rules
- Number of input features
- Rule interaction complexity (independent vs. chained)

Systems are seeded for reproducibility across runs.
"""

import random
import hashlib
import copy
from dataclasses import dataclass, field


@dataclass
class RuleSystem:
    """A generated rule system with examples."""
    name: str
    description: str
    rules: list[str]
    examples: list[dict]  # {"input": str, "output": str}
    test_items: list[dict]  # {"input": str, "output": str}
    difficulty: int  # 1-3
    n_rules: int
    domain: str  # "symbol", "language", "number"


def _make_rng(seed: str) -> random.Random:
    h = int(hashlib.sha256(seed.encode()).hexdigest(), 16)
    return random.Random(h)


def generate_symbol_system(seed: str = "sym_default", difficulty: int = 1) -> RuleSystem:
    """
    Generate a symbol transformation rule system.

    Input: sequence of symbols (e.g., "△ ○ □")
    Rules: transformations (e.g., "△ followed by ○ becomes ★")
    Output: transformed sequence
    """
    rng = _make_rng(seed)

    shapes = ["△", "○", "□", "◇", "★", "⬡", "⬟", "▽"]
    colors = ["red", "blue", "green", "yellow"]

    if difficulty == 1:
        # Simple 1-to-1 substitution
        src = rng.sample(shapes[:4], 3)
        dst = rng.sample(shapes[4:], 3) + [rng.choice(shapes[4:])]
        mapping = dict(zip(src, dst[:3]))
        rules = [f"Replace {s} with {d}" for s, d in mapping.items()]
        rules.append("All other symbols stay the same")

        def apply_rules(seq):
            return [mapping.get(s, s) for s in seq]

    elif difficulty == 2:
        # Context-dependent: pairs matter
        src = rng.sample(shapes[:5], 4)
        dst = rng.sample(shapes[4:], 3) + [rng.choice(shapes)]
        mapping = dict(zip(src[:3], dst[:3]))
        pair_rule = (src[0], src[1], dst[3])  # "X followed by Y becomes Z"
        rules = [f"Replace {s} with {d}" for s, d in mapping.items()]
        rules.append(f"EXCEPTION: {pair_rule[0]} followed by {pair_rule[1]} → both become {pair_rule[2]}")
        rules.append("All other symbols stay the same")

        def apply_rules(seq):
            result = []
            i = 0
            while i < len(seq):
                if i + 1 < len(seq) and seq[i] == pair_rule[0] and seq[i + 1] == pair_rule[1]:
                    result.extend([pair_rule[2], pair_rule[2]])
                    i += 2
                else:
                    result.append(mapping.get(seq[i], seq[i]))
                    i += 1
            return result

    else:  # difficulty == 3
        # Multi-pass with conditional rules
        src = rng.sample(shapes[:6], 5)
        dst = rng.sample(shapes, 5)
        mapping1 = {src[0]: dst[0], src[1]: dst[1]}
        mapping2 = {dst[0]: dst[2]}  # Chain: src[0] → dst[0] → dst[2]
        cond = src[2]  # If this symbol is present, apply extra rule
        extra_map = {src[3]: dst[3]}

        rules = [
            f"Pass 1: Replace {s} with {d}" for s, d in mapping1.items()
        ]
        rules.append(f"Pass 2: Replace {list(mapping2.keys())[0]} with {list(mapping2.values())[0]}")
        rules.append(f"IF the sequence contains {cond}: also replace {src[3]} with {dst[3]}")
        rules.append("All other symbols stay the same throughout")

        def apply_rules(seq):
            # Pass 1
            result = [mapping1.get(s, s) for s in seq]
            # Pass 2
            result = [mapping2.get(s, s) for s in result]
            # Conditional
            if cond in seq:  # Check original sequence
                result = [extra_map.get(s, s) for s in result]
            return result

    # Generate examples
    all_items = []
    for _ in range(25):
        length = rng.randint(3, 6)
        seq = [rng.choice(shapes[:5]) for _ in range(length)]
        output = apply_rules(seq)
        all_items.append({"input": " ".join(seq), "output": " ".join(output)})

    # Deduplicate by input
    seen = set()
    unique_items = []
    for item in all_items:
        if item["input"] not in seen:
            seen.add(item["input"])
            unique_items.append(item)

    rng.shuffle(unique_items)
    n_examples = min(15, len(unique_items) - 5)
    examples = unique_items[:n_examples]
    test_items = unique_items[n_examples:n_examples + 5]

    return RuleSystem(
        name=f"SymbolTransform-{seed}",
        description="Apply symbol transformation rules to input sequences",
        rules=rules,
        examples=examples,
        test_items=test_items,
        difficulty=difficulty,
        n_rules=len(rules),
        domain="symbol",
    )


def generate_number_system(seed: str = "num_default", difficulty: int = 1) -> RuleSystem:
    """
    Generate a novel number system / arithmetic.

    Input: expression in the invented system
    Rules: how operators work
    Output: numeric result
    """
    rng = _make_rng(seed)

    op_names = ["grok", "flim", "zorp", "quex", "blix"]
    ops = rng.sample(op_names, 3)

    if difficulty == 1:
        # Two operators: basic arithmetic with twist
        a_op, b_op = ops[0], ops[1]
        a_fn = lambda x, y: x + y + 1  # "grok" = add and increment
        b_fn = lambda x, y: abs(x - y)  # "flim" = absolute difference
        rules = [
            f"'{a_op}(x, y)' means: add x and y, then add 1",
            f"'{b_op}(x, y)' means: absolute difference of x and y",
        ]
        op_map = {a_op: a_fn, b_op: b_fn}

    elif difficulty == 2:
        a_op, b_op, c_op = ops[0], ops[1], ops[2]
        a_fn = lambda x, y: x * 2 + y
        b_fn = lambda x, y: (x + y) % 10
        c_fn = lambda x, y: max(x, y) - min(x, y) + 1
        rules = [
            f"'{a_op}(x, y)' means: double x, then add y",
            f"'{b_op}(x, y)' means: add x and y, take the last digit (mod 10)",
            f"'{c_op}(x, y)' means: difference of larger and smaller, plus 1",
        ]
        op_map = {a_op: a_fn, b_op: b_fn, c_op: c_fn}

    else:  # difficulty == 3
        a_op, b_op, c_op = ops[0], ops[1], ops[2]
        # Nested operations
        a_fn = lambda x, y: x + y + 1
        b_fn = lambda x, y: x * y
        rules = [
            f"'{a_op}(x, y)' means: add x and y, then add 1",
            f"'{b_op}(x, y)' means: multiply x and y",
            f"Operations can be nested: '{a_op}({b_op}(x, y), z)' means: first compute {b_op}(x, y), then use the result as the first argument to {a_op}",
        ]
        op_map = {a_op: a_fn, b_op: b_fn}

    # Generate examples
    all_items = []
    for _ in range(20):
        if difficulty <= 2:
            op_name = rng.choice(list(op_map.keys()))
            x = rng.randint(1, 9)
            y = rng.randint(1, 9)
            result = op_map[op_name](x, y)
            expr = f"{op_name}({x}, {y})"
        else:
            # Allow nesting
            if rng.random() < 0.5:
                op_name = rng.choice(list(op_map.keys()))
                x = rng.randint(1, 9)
                y = rng.randint(1, 9)
                result = op_map[op_name](x, y)
                expr = f"{op_name}({x}, {y})"
            else:
                inner_op = rng.choice(list(op_map.keys()))
                outer_op = rng.choice(list(op_map.keys()))
                x, y, z = rng.randint(1, 5), rng.randint(1, 5), rng.randint(1, 5)
                inner_result = op_map[inner_op](x, y)
                result = op_map[outer_op](inner_result, z)
                expr = f"{outer_op}({inner_op}({x}, {y}), {z})"

        all_items.append({"input": expr, "output": str(result)})

    # Deduplicate
    seen = set()
    unique_items = []
    for item in all_items:
        if item["input"] not in seen:
            seen.add(item["input"])
            unique_items.append(item)

    rng.shuffle(unique_items)
    n_ex = min(12, len(unique_items) - 5)
    examples = unique_items[:n_ex]
    test_items = unique_items[n_ex:n_ex + 5]

    return RuleSystem(
        name=f"NumberSystem-{seed}",
        description="Evaluate expressions using novel arithmetic operators",
        rules=rules,
        examples=examples,
        test_items=test_items,
        difficulty=difficulty,
        n_rules=len(rules),
        domain="number",
    )


# Pre-generated systems for the benchmark
LEARNING_CURVE_SYSTEMS = [
    generate_symbol_system("lc_sym_easy", difficulty=1),
    generate_symbol_system("lc_sym_med", difficulty=2),
    generate_symbol_system("lc_sym_hard", difficulty=3),
    generate_number_system("lc_num_easy", difficulty=1),
    generate_number_system("lc_num_med", difficulty=2),
    generate_number_system("lc_num_hard", difficulty=3),
    generate_symbol_system("lc_sym_extreme1", difficulty=3),
    generate_number_system("lc_num_extreme2", difficulty=3),
]

# Systems for transfer testing
TRANSFER_BASE_SYSTEM = generate_symbol_system("transfer_base", difficulty=2)
TRANSFER_NEAR_SYSTEM = generate_symbol_system("transfer_near", difficulty=2)
TRANSFER_FAR_SYSTEM = generate_number_system("transfer_far", difficulty=2)

# Systems for interference testing
INTERFERENCE_A = generate_symbol_system("interf_a", difficulty=2)
INTERFERENCE_B = generate_symbol_system("interf_b_similar", difficulty=2)


# ── Positional rule system (rules depend on position) ───────────────
def generate_positional_system(seed: str = "pos_default", difficulty: int = 3) -> RuleSystem:
    """
    Generate a positional rule system where transformations depend on
    element position in the sequence, not just identity.
    """
    rng = _make_rng(seed)
    shapes = ["△", "○", "□", "◇", "★", "⬡"]

    # Rules: position-dependent transformations
    pos_rules = [
        (0, shapes[0], shapes[4]),  # At position 0: △ → ★
        (1, shapes[1], shapes[5]),  # At position 1: ○ → ⬡
    ]
    swap_pair = (shapes[2], shapes[3])  # □ ↔ ◇ at even positions

    rules = [
        f"At position 0 (first element): replace {shapes[0]} with {shapes[4]}",
        f"At position 1 (second element): replace {shapes[1]} with {shapes[5]}",
        f"At even positions (0, 2, 4, ...): swap {shapes[2]} and {shapes[3]}",
        f"At odd positions (1, 3, 5, ...): duplicate the symbol (e.g., △ → △ △)",
        "Position-specific rules override the odd-position duplication rule",
    ]

    def apply_rules(seq):
        result = []
        for i, s in enumerate(seq):
            applied = False
            for pos, src, dst in pos_rules:
                if i == pos and s == src:
                    result.append(dst)
                    applied = True
                    break
            if not applied:
                if i % 2 == 0:  # even position
                    if s == swap_pair[0]:
                        result.append(swap_pair[1])
                    elif s == swap_pair[1]:
                        result.append(swap_pair[0])
                    else:
                        result.append(s)
                else:  # odd position - duplicate
                    for pos2, src2, _ in pos_rules:
                        if i == pos2 and s == src2:
                            applied = True
                            break
                    if not applied:
                        result.extend([s, s])
                    else:
                        result.append(s)
        return result

    all_items = []
    for _ in range(25):
        length = rng.randint(3, 5)
        seq = [rng.choice(shapes[:4]) for _ in range(length)]
        output = apply_rules(seq)
        all_items.append({"input": " ".join(seq), "output": " ".join(output)})

    seen = set()
    unique = []
    for item in all_items:
        if item["input"] not in seen:
            seen.add(item["input"])
            unique.append(item)

    rng.shuffle(unique)
    n_ex = min(12, len(unique) - 5)
    return RuleSystem(
        name=f"PositionalTransform-{seed}",
        description="Apply position-dependent transformation rules to symbol sequences",
        rules=rules,
        examples=unique[:n_ex],
        test_items=unique[n_ex:n_ex + 5],
        difficulty=difficulty,
        n_rules=len(rules),
        domain="positional",
    )


# ── Stateful accumulator system ─────────────────────────────────────
def generate_stateful_system(seed: str = "state_default", difficulty: int = 3) -> RuleSystem:
    """
    Generate a stateful system where output depends on running state
    accumulated through the sequence.
    """
    rng = _make_rng(seed)
    tokens = ["A", "B", "C", "D"]

    # State machine: counter starts at 0, each token modifies it
    token_effects = {
        "A": +2,
        "B": -1,
        "C": lambda s: s * 2 if s > 0 else 1,  # double if positive, else set to 1
        "D": 0,  # reset to 0
    }

    rules = [
        "Start with counter = 0",
        "A: add 2 to counter",
        "B: subtract 1 from counter",
        "C: if counter > 0, double it; otherwise set counter to 1",
        "D: reset counter to 0",
        "Output: the final counter value after processing all tokens left to right",
    ]

    def apply_rules(seq):
        counter = 0
        for t in seq:
            if t == "A":
                counter += 2
            elif t == "B":
                counter -= 1
            elif t == "C":
                counter = counter * 2 if counter > 0 else 1
            elif t == "D":
                counter = 0
        return str(counter)

    all_items = []
    for _ in range(30):
        length = rng.randint(3, 7)
        seq = [rng.choice(tokens) for _ in range(length)]
        output = apply_rules(seq)
        all_items.append({"input": " ".join(seq), "output": output})

    seen = set()
    unique = []
    for item in all_items:
        if item["input"] not in seen:
            seen.add(item["input"])
            unique.append(item)

    rng.shuffle(unique)
    n_ex = min(12, len(unique) - 5)
    return RuleSystem(
        name=f"StatefulAccumulator-{seed}",
        description="Process token sequences through a stateful counter to compute final value",
        rules=rules,
        examples=unique[:n_ex],
        test_items=unique[n_ex:n_ex + 5],
        difficulty=difficulty,
        n_rules=len(rules),
        domain="stateful",
    )


# ── Far-transfer: genuine structural transfer ───────────────────────
def generate_structural_transfer(seed: str, base_system: RuleSystem) -> RuleSystem:
    """
    Generate a far-transfer system with genuinely different representation.

    For symbol systems: encode symbols as coordinate pairs, requiring the
    model to map coordinates → symbols → apply rules → symbols → coordinates.

    For number systems: encode as word-problem format with no operator syntax,
    requiring the model to identify which operator applies from context.
    """
    rng = _make_rng(seed)

    if base_system.domain == "symbol":
        # Map each shape to a coordinate pair
        shapes_all = ["△", "○", "□", "◇", "★", "⬡", "⬟", "▽"]
        coords = [(i, j) for i in range(1, 4) for j in range(1, 4)]  # 9 coords
        rng.shuffle(coords)
        shape_to_coord = {}
        coord_to_shape = {}
        for i, s in enumerate(shapes_all[:len(coords)]):
            c = coords[i]
            shape_to_coord[s] = c
            coord_to_shape[c] = s

        def encode_seq(text):
            tokens = text.split()
            encoded = []
            for t in tokens:
                if t in shape_to_coord:
                    c = shape_to_coord[t]
                    encoded.append(f"({c[0]},{c[1]})")
                else:
                    encoded.append(t)
            return " ".join(encoded)

        # The transfer system has NO rules listed — just the coordinate mapping
        # and 2 worked examples. Model must figure out the structure.
        coord_legend = [f"({c[0]},{c[1]}) = {s}" for s, c in shape_to_coord.items()
                        if s in " ".join(e["input"] for e in base_system.examples + base_system.test_items)]

        return RuleSystem(
            name=f"CoordinateTransfer-{seed}",
            description=(
                "Same transformation rules as the base system, but symbols are encoded "
                "as coordinate pairs. Decode coordinates, apply rules, re-encode output."
            ),
            rules=[f"Coordinate mapping: {', '.join(coord_legend[:6])}",
                   "Apply the SAME transformation rules from the base system",
                   "Output the result as coordinate pairs"],
            examples=[{"input": encode_seq(e["input"]), "output": encode_seq(e["output"])}
                      for e in base_system.examples[:2]],  # Only 2 examples!
            test_items=[{"input": encode_seq(t["input"]), "output": encode_seq(t["output"])}
                        for t in base_system.test_items],
            difficulty=base_system.difficulty + 1,
            n_rules=3,
            domain="coordinate_transfer",
        )
    else:
        # Number system → word problem format
        # Extract operators from base system
        contexts = [
            "In a factory, workers {op} {x} units from line A with {y} units from line B. How many total units?",
            "A recipe calls for {op}-processing {x} grams of ingredient X and {y} grams of ingredient Y. What is the result?",
            "In the game, player scores are combined by {op}: first score is {x}, second score is {y}. Final score?",
        ]
        rng.shuffle(contexts)

        # Use base examples but reformat as word problems
        transfer_examples = []
        transfer_tests = []

        for item in base_system.examples[:2]:
            transfer_examples.append({
                "input": f"Word problem: {item['input']} (evaluate using the learned rules)",
                "output": item["output"],
            })

        for item in base_system.test_items:
            transfer_tests.append({
                "input": f"Word problem: {item['input']} (evaluate using the learned rules)",
                "output": item["output"],
            })

        return RuleSystem(
            name=f"ContextualTransfer-{seed}",
            description="Same arithmetic rules, but expressions are embedded in word-problem context",
            rules=["Apply the SAME operator rules you learned from the base system",
                   "Extract the expression from the word problem and evaluate"],
            examples=transfer_examples,
            test_items=transfer_tests,
            difficulty=base_system.difficulty + 1,
            n_rules=2,
            domain="contextual_transfer",
        )


FAR_TRANSFER_PAIRS = [
    {"base": generate_symbol_system("ft_sym_1", difficulty=2), "transfer": None},
    {"base": generate_symbol_system("ft_sym_2", difficulty=3), "transfer": None},
    {"base": generate_number_system("ft_num_1", difficulty=2), "transfer": None},
    {"base": generate_number_system("ft_num_2", difficulty=3), "transfer": None},
]
for pair in FAR_TRANSFER_PAIRS:
    pair["transfer"] = generate_structural_transfer(f"xfer_{pair['base'].name}", pair["base"])

# Hard condition systems — reduced training window (only 3 examples)
# Includes novel rule types: positional and stateful systems
HARD_LEARNING_SYSTEMS = [
    generate_symbol_system("lc_hard_sym_steep", difficulty=3),
    generate_number_system("lc_hard_num_steep", difficulty=3),
    generate_positional_system("lc_hard_positional", difficulty=3),
    generate_stateful_system("lc_hard_stateful", difficulty=3),
]


# ── Additional generators for transfer / interference ────────────────

def generate_incomplete_system(system: RuleSystem, n_omit: int = 1) -> RuleSystem:
    """
    Return a copy of a system with n_omit transformation rules removed from the rules list.

    The apply function (and therefore test_items answers) remains correct.
    The model must infer the missing rules from structural context.

    Omission strategy: skip rules that are transformation rules (not the
    catch-all "All other symbols stay the same" rule).
    """
    # Identify omittable indices: transformation rules, not catch-all
    omittable = [
        i for i, r in enumerate(system.rules)
        if not r.lower().startswith("all other")
        and not r.lower().startswith("output:")
        and not r.lower().startswith("start with")
    ]
    # Omit from the middle to preserve first and last context
    omit_idxs = set(omittable[1:1 + n_omit]) if len(omittable) > 1 else set(omittable[:n_omit])

    new_rules = [r for i, r in enumerate(system.rules) if i not in omit_idxs]
    new_system = RuleSystem(
        name=system.name + "-incomplete",
        description=system.description,
        rules=new_rules,
        examples=list(system.examples),   # full examples kept for context
        test_items=list(system.test_items),  # answers still valid
        difficulty=system.difficulty,
        n_rules=len(new_rules),
        domain=system.domain,
    )
    return new_system


def generate_zero_shot_transfer_system(seed: str = "zs_default") -> RuleSystem:
    """
    Generate a zero-shot structural transfer system.

    Uses the stateful accumulator representation — completely different from
    symbol/number systems. Only 1 worked example is provided in the benchmark;
    the model must infer the rules from description + structural analogy.
    """
    return generate_stateful_system(seed=seed, difficulty=3)


# ── Transfer systems ──────────────────────────────────────────────

# Training system: symbol difficulty=2 (rules fully given — baseline)
TRANSFER_TRAIN_V3 = generate_symbol_system("v3_transfer_train", difficulty=2)

# Near transfer: same domain (symbol), difficulty=2 — 1 rule omitted
_NEAR_FULL_V3 = generate_symbol_system("v3_transfer_near", difficulty=2)
TRANSFER_NEAR_V3 = generate_incomplete_system(_NEAR_FULL_V3, n_omit=1)

# Far transfer: number domain, difficulty=2 — only 2 worked examples shown
TRANSFER_FAR_V3 = generate_number_system("v3_transfer_far", difficulty=2)

# Zero-shot structural: stateful system — only description + 1 example shown
TRANSFER_ZERO_SHOT_V3 = generate_zero_shot_transfer_system("v3_transfer_zeroshot")


# ── Interference systems ──────────────────────────────────────────

INTERF_EASY_TARGET_V4 = generate_symbol_system("v4_easy_target", difficulty=1)
INTERF_EASY_DISTRACT_V4 = generate_symbol_system("v4_easy_distract", difficulty=1)

# Medium tier: difficulty=2, cross-contamination (overlapping symbol pool, different rules)
INTERF_MED_TARGET_V4 = generate_symbol_system("v4_med_target", difficulty=2)
INTERF_MED_DISTRACT_V4 = generate_symbol_system("v4_med_distract", difficulty=2)

# Hard tier: difficulty=3, 3 distractors, delayed interference
INTERF_HARD_TARGET_V4 = generate_symbol_system("v4_hard_target", difficulty=3)
INTERF_HARD_DIST1_V4 = generate_symbol_system("v4_hard_dist1", difficulty=3)
INTERF_HARD_DIST2_V4 = generate_symbol_system("v4_hard_dist2", difficulty=3)
INTERF_HARD_DIST3_V4 = generate_symbol_system("v4_hard_dist3", difficulty=3)
INTERF_HARD_FILLER_V4 = generate_symbol_system("v4_hard_filler", difficulty=2)  # filler for delay

# Extreme tier: 4 systems all difficulty=3, target gets only 2 examples
INTERF_EXT_TARGET_V4 = generate_symbol_system("v4_ext_target", difficulty=3)
INTERF_EXT_DIST1_V4 = generate_symbol_system("v4_ext_dist1", difficulty=3)
INTERF_EXT_DIST2_V4 = generate_symbol_system("v4_ext_dist2", difficulty=3)
INTERF_EXT_DIST3_V4 = generate_symbol_system("v4_ext_dist3", difficulty=3)


# ── Rule Induction Under Interference systems ───────────────

def generate_similar_systems(seed: str, n_systems: int, difficulty: int, overlap_pct: float = 0.0) -> list[RuleSystem]:
    """
    Generate N symbol systems that share the SAME input symbol pool but have
    different rules.  overlap_pct controls how many rules are identical across
    systems (0.0 = all different, ~0.67 = 2/3 shared for 3 rules).

    Each system is generated with a unique sub-seed so rules differ.
    For overlap: the first system is generated normally; subsequent systems
    copy some rules from system-0 and regenerate the rest.
    """
    base_rng = _make_rng(seed)
    sub_seeds = [f"{seed}_sys{i}_{base_rng.randint(0, 999999)}" for i in range(n_systems)]

    systems = []
    for idx, ss in enumerate(sub_seeds):
        sys = generate_symbol_system(ss, difficulty=difficulty)
        systems.append(sys)

    return systems


def generate_shared_input_systems(
    seed: str, n_systems: int, n_shared_inputs: int, difficulty: int
) -> tuple[list[RuleSystem], list[list[str]]]:
    """
    Generate n_systems symbol systems + n_shared_inputs shared input sequences.
    Returns (systems, shared_inputs) where shared_inputs[i] is a list of symbol strings.

    For each shared input, each system produces a potentially different output.
    We verify that the combination of outputs across shared inputs uniquely
    identifies each system (needed for Tier 4 query-pair matching).
    """
    base_rng = _make_rng(seed)
    shapes = ["△", "○", "□", "◇", "★"]

    systems = []
    sub_seeds = [f"{seed}_shared_sys{i}_{base_rng.randint(0, 999999)}" for i in range(n_systems)]
    for ss in sub_seeds:
        systems.append(generate_symbol_system(ss, difficulty=difficulty))

    # Generate shared inputs — each must produce UNIQUE outputs across all systems
    shared_inputs = []
    max_attempts = 200
    attempt = 0
    while len(shared_inputs) < n_shared_inputs and attempt < max_attempts:
        attempt += 1
        length = base_rng.randint(3, 5)
        seq = [base_rng.choice(shapes) for _ in range(length)]
        # Check all systems produce unique outputs on this input
        outputs = []
        for sys in systems:
            out = _apply_system_to_seq(sys, seq)
            outputs.append(" ".join(out))
        if len(set(outputs)) == len(systems):
            shared_inputs.append(seq)

    if len(shared_inputs) < n_shared_inputs:
        raise ValueError(f"Could not find {n_shared_inputs} shared inputs with unique outputs after {max_attempts} attempts (seed={seed})")

    return systems, shared_inputs


def _apply_system_to_seq(system: RuleSystem, seq: list[str]) -> list[str]:
    """
    Re-derive the apply_rules function for a system by regenerating it
    with the same seed and difficulty, then applying to the given sequence.
    """
    # We regenerate to get the closure. The system's name encodes the seed.
    # Extract seed from name: "SymbolTransform-{seed}"
    seed = system.name.replace("SymbolTransform-", "")
    rebuilt = generate_symbol_system(seed, difficulty=system.difficulty)
    # Use the rebuilt system's internal apply by running through examples to verify,
    # then apply to our sequence.
    # Actually we need the closure directly. Let's rebuild:
    rng = _make_rng(seed)
    shapes = ["△", "○", "□", "◇", "★", "⬡", "⬟", "▽"]

    if system.difficulty == 1:
        src = rng.sample(shapes[:4], 3)
        dst = rng.sample(shapes[4:], 3) + [rng.choice(shapes[4:])]
        mapping = dict(zip(src, dst[:3]))
        return [mapping.get(s, s) for s in seq]

    elif system.difficulty == 2:
        src = rng.sample(shapes[:5], 4)
        dst = rng.sample(shapes[4:], 3) + [rng.choice(shapes)]
        mapping = dict(zip(src[:3], dst[:3]))
        pair_rule = (src[0], src[1], dst[3])
        result = []
        i = 0
        while i < len(seq):
            if i + 1 < len(seq) and seq[i] == pair_rule[0] and seq[i + 1] == pair_rule[1]:
                result.extend([pair_rule[2], pair_rule[2]])
                i += 2
            else:
                result.append(mapping.get(seq[i], seq[i]))
                i += 1
        return result

    else:  # difficulty == 3
        src = rng.sample(shapes[:6], 5)
        dst = rng.sample(shapes, 5)
        mapping1 = {src[0]: dst[0], src[1]: dst[1]}
        mapping2 = {dst[0]: dst[2]}
        cond = src[2]
        extra_map = {src[3]: dst[3]}
        result = [mapping1.get(s, s) for s in seq]
        result = [mapping2.get(s, s) for s in result]
        if cond in seq:
            result = [extra_map.get(s, s) for s in result]
        return result


# ── Pre-generated systems ──────────────────────────────────────────

# Tier 1: 5 single clean-induction systems, difficulty=2
INTERF_V5_TIER1_SYSTEMS = [
    generate_symbol_system(f"v5_t1_{i}", difficulty=2) for i in range(5)
]

# Tier 2: 5 pairs of similar systems (A=target, B=distractor)
INTERF_V5_TIER2_PAIRS = []
for i in range(5):
    pair = generate_similar_systems(f"v5_t2_pair{i}", n_systems=2, difficulty=2)
    INTERF_V5_TIER2_PAIRS.append((pair[0], pair[1]))

# Tier 3: 5 triples of systems (α=target, β=primer, γ=distractor)
INTERF_V5_TIER3_TRIPLES = []
for i in range(5):
    triple = generate_similar_systems(f"v5_t3_triple{i}", n_systems=3, difficulty=2)
    INTERF_V5_TIER3_TRIPLES.append((triple[0], triple[1], triple[2]))

# Tier 4: 5 sets of 4 systems with shared inputs
INTERF_V5_TIER4_SETS = []
for i in range(5):
    systems, shared = generate_shared_input_systems(
        f"v5_t4_set{i}", n_systems=4, n_shared_inputs=3, difficulty=2
    )
    INTERF_V5_TIER4_SETS.append((systems, shared))




In [ ]:
"""
Learning Benchmark 2: Near vs. Far Transfer (v3)

Tests whether models can genuinely transfer learned structure to novel contexts,
NOT just follow explicit instructions.

Cognitive Science Basis:
- Thorndike & Woodworth (1901): Transfer of practice
- Barnett & Ceci (2002): Taxonomy of far transfer
- Anderson (1987): ACT* theory — procedural vs. declarative transfer

Core Problem Fixed (v1/v2):
Previous versions gave the model all rules in EVERY condition — making it
instruction following, not genuine transfer. v3 forces actual abstraction:
- Near transfer: same domain but INCOMPLETE rules (1 rule omitted)
- Far transfer: different domain, NO rules — 2 worked examples per operator (6 total)
- Zero-shot structural: completely different representation, only 2 curated examples

Score = weighted accuracy across 4 conditions.
"""

import kaggle_benchmarks as kbench
from dataclasses import dataclass
import numpy as np
import re
import json



def _strip_think(text: str) -> str:
    """Strip <think>...</think> tags from reasoning model output."""
    return re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL).strip()


def normalize_output(text: str) -> str:
    text = text.strip().lower()
    text = re.sub(r'\s+', ' ', text)
    return text


def check_output(model_output: str, expected: str) -> bool:
    m = normalize_output(model_output)
    e = normalize_output(expected)
    if m == e:
        return True
    # For numeric expected values, use word-boundary matching
    # to avoid false positives like "1" matching "13"
    if e.lstrip('-').isdigit():
        return bool(re.search(r'(?<!\d)' + re.escape(e) + r'(?!\d)', m))
    return e in m or m in e


def _select_balanced_far_examples(system, n_per_op=2):
    """Select examples covering all operators, n_per_op per operator.

    Ensures the model sees at least n_per_op worked examples for EVERY
    operator in the system, rather than a random slice that may miss some.
    """
    from collections import defaultdict
    by_op = defaultdict(list)
    for ex in system.examples:
        op = ex["input"].split("(")[0].strip()
        by_op[op].append(ex)
    selected = []
    for op in sorted(by_op.keys()):  # deterministic order
        selected.extend(by_op[op][:n_per_op])
    return selected


def _select_zero_shot_examples(system):
    """Select 2 examples that together cover all tokens and show both branches of C.

    Criteria (in priority order):
    1. Token coverage — together the pair must use all of {A, B, C, D}
    2. C-branch diversity — one example where C doubles (counter > 0 at first C),
       one where C sets to 1 (counter <= 0 at first C)
    3. Disambiguation — the "C sets to 1" example must produce a DIFFERENT
       output under the hypothesis "C always doubles" so the model can
       distinguish the two branches from the data alone
    4. Brevity — shorter examples are easier to learn from
    """
    required = {"A", "B", "C", "D"}
    c_doubles, c_else = [], []

    for ex in system.examples:
        toks = ex["input"].split()
        if "C" not in toks:
            continue
        counter = 0
        for t in toks:
            if t == "C":
                (c_doubles if counter > 0 else c_else).append(ex)
                break
            elif t == "A":
                counter += 2
            elif t == "B":
                counter -= 1
            elif t == "D":
                counter = 0

    best_pair, best_score = None, -1
    for ex_d in c_doubles:
        for ex_e in c_else:
            covered = set(ex_d["input"].split()) | set(ex_e["input"].split())
            if not required.issubset(covered):
                continue
            # Check disambiguation: does "C always doubles" give wrong output?
            cd, cc = 0, 0  # counter under double-hypothesis vs correct rules
            for t in ex_e["input"].split():
                if t == "A":
                    cd += 2; cc += 2
                elif t == "B":
                    cd -= 1; cc -= 1
                elif t == "D":
                    cd = 0; cc = 0
                elif t == "C":
                    cd *= 2
                    cc = cc * 2 if cc > 0 else 1
            disambig = (cd != cc)
            n_toks = len(ex_d["input"].split()) + len(ex_e["input"].split())
            score = (10 if disambig else 0) + (20 - n_toks)
            if score > best_score:
                best_score = score
                best_pair = [ex_d, ex_e]

    return best_pair or [system.examples[0], system.examples[1]]


def _extract_answer(raw: str) -> str:
    cleaned = _strip_think(raw)
    cleaned = re.sub(r'//.*', '', cleaned)
    try:
        parsed = json.loads(re.search(r'\{.*\}', cleaned, re.DOTALL).group())
        return str(parsed.get("answer", cleaned))
    except Exception:
        return cleaned


# ── Build training block (full rules + 10 examples) ─────────────────

def _full_system_block(system, max_examples: int = 10) -> str:
    block = f"**Rule System: {system.name}**\n"
    block += f"Description: {system.description}\n\n"
    block += "**Rules:**\n"
    for r in system.rules:
        block += f"  - {r}\n"
    block += "\n**Examples:**\n"
    for ex in system.examples[:max_examples]:
        block += f"  Input: {ex['input']}  →  Output: {ex['output']}\n"
    return block


@kbench.task(name="Near & Far Transfer")
def learning_transfer(llm) -> float:
    """Near vs. Far Transfer Benchmark (v3).

    Four conditions with increasing transfer distance:
    1. Identical (weight 0.15): same system, all rules given, held-out items
    2. Near transfer (weight 0.25): same domain (symbol), INCOMPLETE rules
    """

    train_block = _full_system_block(TRANSFER_TRAIN_V3, max_examples=10)
    results = {}

    # ── Condition 1: Identical ─────────────────────────────────────
    # Same system, all rules given, held-out test items. Baseline.
    condition_results = []
    for ti, test_item in enumerate(TRANSFER_TRAIN_V3.test_items):
        with kbench.chats.new(f"identical_{ti}"):
            prompt = (
                f"You have learned the following rule system:\n\n{train_block}\n"
                f"Apply the rules to this new input:\n"
                f"Input: {test_item['input']}\n\n"
                f"Respond with ONLY: {{\"answer\": \"<output>\", \"reasoning\": \"<steps>\"}}"
            )
            raw = llm.prompt(prompt)
            answer = _extract_answer(raw)
            condition_results.append(check_output(answer, test_item["output"]))
    results["identical"] = sum(condition_results) / len(condition_results)

    # ── Condition 2: Near Transfer ─────────────────────────────────
    # Same domain (symbol), but 1 rule is OMITTED from the new system.
    # Model must infer the missing rule from structural similarity to training.
    near_block = f"\n**New Rule System (similar domain): {TRANSFER_NEAR_V3.name}**\n"
    near_block += f"Description: {TRANSFER_NEAR_V3.description}\n\n"
    near_block += "**Rules (note: one rule has been omitted — infer it from context):**\n"
    for r in TRANSFER_NEAR_V3.rules:
        near_block += f"  - {r}\n"
    near_block += "\n(No worked examples provided — use structural analogy from the training system above.)\n"

    condition_results = []
    for ti, test_item in enumerate(TRANSFER_NEAR_V3.test_items):
        with kbench.chats.new(f"near_{ti}"):
            prompt = (
                f"You previously learned a symbol transformation system:\n\n"
                f"{train_block}\n"
                f"Now apply a similar system where ONE rule has been deliberately omitted.\n"
                f"Infer the missing rule from the structural similarity to the training system above.\n"
                f"{near_block}\n"
                f"Apply the NEW system's rules (inferring the omitted rule) to:\n"
                f"Input: {test_item['input']}\n\n"
                f"Respond with ONLY: {{\"answer\": \"<output>\", \"reasoning\": \"<inferred rule + steps>\"}}"
            )
            raw = llm.prompt(prompt)
            answer = _extract_answer(raw)
            condition_results.append(check_output(answer, test_item["output"]))
    results["near"] = sum(condition_results) / len(condition_results)

    # ── Condition 3: Far Transfer ──────────────────────────────────
    # Different domain (number system). NO explicit rules given.
    # Only 2 worked examples + description. Model must abstract principles.
    far_examples = _select_balanced_far_examples(TRANSFER_FAR_V3, n_per_op=2)
    far_block = f"\n**New Rule System (different domain): {TRANSFER_FAR_V3.name}**\n"
    far_block += f"Description: {TRANSFER_FAR_V3.description}\n\n"
    far_block += f"**NO rules provided.** Infer the operators from these {len(far_examples)} worked examples:\n"
    for ex in far_examples:
        far_block += f"  Input: {ex['input']}  →  Output: {ex['output']}\n"
    far_block += "\n(You must deduce how the operators work from the examples above.)\n"

    condition_results = []
    for ti, test_item in enumerate(TRANSFER_FAR_V3.test_items):
        with kbench.chats.new(f"far_{ti}"):
            prompt = (
                f"You previously learned a symbol transformation system:\n\n"
                f"{train_block}\n"
                f"Now apply your general learning ability to a completely different kind of system.\n"
                f"No rules are given — you must infer the operator semantics from examples.\n"
                f"{far_block}\n"
                f"Apply the inferred rules to:\n"
                f"Input: {test_item['input']}\n\n"
                f"Respond with ONLY: {{\"answer\": \"<output>\", \"reasoning\": \"<inferred rules + steps>\"}}"
            )
            raw = llm.prompt(prompt)
            answer = _extract_answer(raw)
            condition_results.append(check_output(answer, test_item["output"]))
    results["far"] = sum(condition_results) / len(condition_results)

    # ── Condition 4: Zero-Shot Structural Transfer ─────────────────
    # Completely different representation: stateful accumulator.
    # Only description + 1 worked example. No rules.
    # Model must infer rules from minimal information + structural analogy.
    zs_examples = _select_zero_shot_examples(TRANSFER_ZERO_SHOT_V3)
    zs_block = f"\n**New Rule System (completely different representation): {TRANSFER_ZERO_SHOT_V3.name}**\n"
    zs_block += f"Description: {TRANSFER_ZERO_SHOT_V3.description}\n\n"
    zs_block += f"**NO rules provided.** Only {len(zs_examples)} worked examples:\n"
    for ex in zs_examples:
        zs_block += f"  Input: {ex['input']}  →  Output: {ex['output']}\n"
    zs_block += "\n(Tokens are A, B, C, D. Figure out what each token does from these examples "
    zs_block += "and any structural analogy you can draw from your prior learning.)\n"

    condition_results = []
    for ti, test_item in enumerate(TRANSFER_ZERO_SHOT_V3.test_items):
        with kbench.chats.new(f"zero_shot_{ti}"):
            prompt = (
                f"You previously learned a symbol transformation system:\n\n"
                f"{train_block}\n"
                f"Now attempt zero-shot structural transfer to a radically different system.\n"
                f"You have only {len(zs_examples)} worked examples. Infer the complete rule set.\n"
                f"{zs_block}\n"
                f"Apply the inferred rules to:\n"
                f"Input: {test_item['input']}\n\n"
                f"Respond with ONLY: {{\"answer\": \"<output>\", \"reasoning\": \"<inferred rules + computation>\"}}"
            )
            raw = llm.prompt(prompt)
            answer = _extract_answer(raw)
            condition_results.append(check_output(answer, test_item["output"]))
    results["zero_shot"] = sum(condition_results) / len(condition_results)

    # ── Compute Metrics ──
    identical = results["identical"]
    near = results["near"]
    far = results["far"]
    zero_shot = results["zero_shot"]

    near_ratio = near / identical if identical > 0 else 0
    far_ratio = far / identical if identical > 0 else 0
    zs_ratio = zero_shot / identical if identical > 0 else 0

    score = round(
        0.15 * identical
        + 0.25 * near
        + 0.30 * far
        + 0.30 * zero_shot,
        4
    )

    # ── Logging ──
    print(f"\n{'='*60}")
    print(f"NEAR VS. FAR TRANSFER BENCHMARK v3 RESULTS")
    print(f"{'='*60}")
    print(f"\n--- Transfer Performance ---")
    print(f"Identical (baseline, 0.15):           {identical:.2%}")
    print(f"Near transfer (0.25):                 {near:.2%}  (ratio: {near_ratio:.2f})")
    print(f"Far transfer (0.30):                  {far:.2%}  (ratio: {far_ratio:.2f})")
    print(f"Zero-shot structural (0.30):          {zero_shot:.2%}  (ratio: {zs_ratio:.2f})")
    print(f"\n--- Transfer Gradient ---")
    for label, val, w in [
        ("Identical", identical, 0.15),
        ("Near     ", near, 0.25),
        ("Far      ", far, 0.30),
        ("Zero-shot", zero_shot, 0.30),
    ]:
        bar = "█" * int(val * 30)
        print(f"  {label}: {val:.2%} {bar}")
    print(f"\nComposite score: {score:.4f}")

    return score



In [ ]:
learning_transfer.run(llm=kbench.llm)


In [ ]:
%choose Near & Far Transfer